In [ ]:
with open("input.txt", "r") as f:
    text = f.read()

In [ ]:
print(text[:1000])
print(len(text))

In [ ]:
chars = sorted(list(set(text))) # remove duplicate chars, convert into a list, order
vocab_size = len(chars) # size of the chars list
print(''.join(chars)) #print chars in a string
print(vocab_size) # print size of the vocab

In [ ]:
#Tokenizing

# char to integer
stoi = {}
for i, ch in enumerate(chars):
    stoi[ch] = i # stoi = {'a': 0, 'b': 1, 'c': 2}

# integer to char lookup
itos = {}
for i, ch in enumerate(chars):
    itos[i] = ch #itos = {0: 'a', 1: 'b', 2: 'c'}

def encode(s):
    result = []
    for c in s:
        result.append(stoi[c]) # appends the equivalent number to the char 
    return result

def decode(l):
    result = []
    for i in l:
        result.append(itos[i])
    return ''.join(result)

print(encode("hi there"))
print(decode(encode("hii there")))

# "hello world" example
# ['d', 'e', 'h', 'l', 'o', 'r', 'w', ' ']
# sorted puts space first (ASCII), then letters alphabetically

# enumerate gives:
# 0 → ' '
# 1 → 'd'
# 2 → 'e'
# 3 → 'h'
# 4 → 'l'
# 5 → 'o'
# 6 → 'r'
# 7 → 'w'

# encode("hello") -> [3,2,4,4,5]

In [ ]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])


In [ ]:
n = int(0.9*len(data))
train_data = data[:n] #90% train
val_data = data[n:] #10% test data

In [ ]:
block_size = 8
train_data[:block_size + 1]


In [ ]:
print(text[:block_size + 1])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size + 1] # + 1 to obtain the target (value that comes next)

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel 
block_size = 8 # what is the maximum context length for predictions
# essentially a batch_size * block_size matrix


def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    if split == 'train':
        data = train_data
    else:
        data = val_data
    
    ix = torch.randint(len(data) - block_size, (batch_size,)) # 4 random positions in the data
    
    x_list = []
    for i in ix:
        chunk = data[i : i + block_size]
        x_list.append(chunk)
    x = torch.stack(x_list)

    y_list = []
    for i in ix:
        chunk = data[i + 1 : i + block_size + 1]
        y_list.append(chunk)
    y = torch.stack(y_list)

    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

In [ ]:
print(xb)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # create a lookup table size vocab_size * vocab_size, every char gets its own embedding during training

    def forward(self, idx, targets=None): #Input batch of size (B, T)

        logits = self.token_embedding_table(idx) # (B,T,C) -> B = 4 sequences, T = 8 positions, C = 65 channels, each channel is now represented as a row of 65 numbers called logits
        
        if targets is None:
            loss = None

        else: 
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # convert the B,T,C into a 2D array
            targets = targets.view(B*T) # convert the targets into a 1D array
            loss = F.cross_entropy(logits, targets) # it takes the logits, softmax converts them to probabilities and finally pushes the -log(probab) to punish low confidence and to obtain the loss value
        '''
        BEFORE
        logits[0][0] = [0.2, 0.8, 0.1, ...]  # 65 values — sequence 0, position 0
        logits[0][1] = [0.5, 0.1, 0.9, ...]  # 65 values — sequence 0, position 1
        logits[1][0] = [0.9, 0.2, 0.4, ...]  # 65 values — sequence 1, position 0
        ...
        AFTER
        row 0  = [0.2, 0.8, 0.1, ...]  # was logits[0][0]
        row 1  = [0.5, 0.1, 0.9, ...]  # was logits[0][1]
        row 8  = [0.9, 0.2, 0.4, ...]  # was logits[1][0]
        '''
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx) # call forward()
            
            # focus only on the last time step
            logits = logits[:, -1, :] # you predict the last position because thats where the next char lives (B, C)
            
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # dim=-1 operates on the last dimension of the tensor (B, C), so in C (across the 65 scores)
            
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) -> this makes that instead of picking the highest probability character, it picks random samples based on the probabilities, so a character with 0.6 probab gets picked more often than one with .1 but not always, in order to make the output more creative
            
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1) concatenate the new character into the end of the sequence
        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1,1), dtype=torch.long) # (1, 1) = [[0]] first char, 0
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))
# predicting chars with generate up until 100 chars. [[0, 23, ... 67]], shape (1, 101)
# taking the first sequence shape (101,), [0] select the list in 1D rather than a nested list, to later decode it
# convert the integers back to characters "hello..."


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32

for steps in range(10000):

    #sample batch of data
    xb, yb = get_batch('train')

    # eval loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

In [ ]:
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

In [ ]:
# Mathemtaical trick in self-attention

torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channels
x = torch.rand(B,T,C)
x.shape

# We need to get tokens to interact with previous tokens
# If I'm at the 5th token we would take the mean of the channels of the 4th tokems, 3rd, 2nd... and obtain a feature vector that summarizes the context history

In [ ]:
xbow = torch.zeros((B,T,C)) # xbow = bag of words

for b in range(B):
    for t in range (T):
        xprev = x[b, :t+1] #(T,C)
        xbow[b,t] = torch.mean(xprev, 0) #time is dimension 0 because B was removed, C would be dimension 1

'''
t=0   xprev = [a]                →  mean([a])           →  xbow[b,0]
t=1   xprev = [a, b]             →  mean([a, b])         →  xbow[b,1]
t=2   xprev = [a, b, c]          →  mean([a, b, c])      →  xbow[b,2]
t=3   xprev = [a, b, c, d]       →  mean([a, b, c, d])   →  xbow[b,3]
...
t=7   xprev = [a, b, c, d, e, f, g, h]  →  mean of all  →  xbow[b,7]
'''

x[0] # will show the first T

In [ ]:
xbow[0] # this will show the tensor that is averaging the previous results in the box of words (bow)

In [ ]:
torch.tril(torch.ones(3,3)) # triangular lower portion of the matrix, tril


In [ ]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True) # ensure that all the elements in a add up to 1
b = torch.randint(0, 10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('---')
print('b=')
print(b)
print('---')
print('c=')
print(c)

In [ ]:
# Method 1
xbow = torch.zeros((B,T,C)) # xbow = bag of words

for b in range(B):
    for t in range (T):
        xprev = x[b, :t+1] #(T,C)
        xbow[b,t] = torch.mean(xprev, 0) #time is dimension 0 because B was removed, C would be dimension 1

'''
t=0   xprev = [a]                →  mean([a])           →  xbow[b,0]
t=1   xprev = [a, b]             →  mean([a, b])         →  xbow[b,1]
t=2   xprev = [a, b, c]          →  mean([a, b, c])      →  xbow[b,2]
t=3   xprev = [a, b, c, d]       →  mean([a, b, c, d])   →  xbow[b,3]
...
t=7   xprev = [a, b, c, d, e, f, g, h]  →  mean of all  →  xbow[b,7]
'''

x[0] # will show the first T

In [ ]:
# Method 2

wei = torch.tril(torch.ones(T, T)) #a token at the T dimenion will only get information from the tokens preceiving it
wei = wei / wei.sum(1, keepdim=True) # we create the matrix in the lower triangle to create a weighted sum
xbow2 = wei @ x #(B, T, T) @ (B, T, C) ---> multiply both matrices to create (B, T, C)

wei

In [ ]:
# Method 3
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T)) #create matrix TxT of zeroes
wei = wei.masked_fill(tril == 0, float('-inf')) # using the initial tril, taking all the values that are 0 and changing them with -inf, this makes it so they cant communicate with tokens in the future
wei = F.softmax(wei, dim = 1) # softmax for every single row to obtain normalized values
xbow3 = wei @ x # multiply it times x to obtain the box of words
torch.allclose(xbow, xbow3)

In [ ]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T)) #create tril of zeroes
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = 1) 
wei


In [ ]:
# Version 4: SELF ATTENTION
# Attention is a communication mechanism, every node has a vector of information, and it aggregates information via a weighted sum of all of the nodes that point to it.
# Attention adds over a set of vectors in the graph
# Scaled attention divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and softmax will stay diffuse and not saturate too much.

torch.manual_seed(1337)
B,T,C = 4, 8, 32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False) # (32,16)
query = nn.Linear(C, head_size, bias=False) # (32,16)
value = nn.Linear(C, head_size, bias=False) # (32,16)

# Every token produces a query and a key by projecting x through a linear layer
k = key(x) # (B, T, 16) --> what am I looking for
q = query(x) # (B, T, 16) --> what do I contain
wei = q @ k.transpose(-2, -1) # we transpose k dim -2 (16) and dim -1 (T) in order to be able to do the dot product. (B, T, 16) @ (B, 16, T) --> (B, T, T)


tril = torch.tril(torch.ones(T,T))
#wei = torch.zeros((T,T)) #create matrix TxT of zeroes
wei = wei.masked_fill(tril == 0, float('-inf')) # using the initial tril, taking all the values that are 0 and changing them with -inf, this makes it so they cant communicate with tokens in the future
wei = F.softmax(wei, dim = 1) # softmax for every single row to obtain normalized values
v = value(x) # (B, T, 16)
out = wei @ v # (B, T, T) @ (B, T, 16) --> (B, T, 16)
#out = wei @ x # multiply it times x to obtain the box of words. X is the information of a token

out.shape

In [ ]:
tril

In [ ]:
wei[0]

In [ ]:
x

In [ ]:
# Scaled attention divides wei by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and softmax will stay diffuse and not saturate too much.

k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

In [ ]:
q.var()

In [ ]:
wei.var()

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean, 1 for rows, 0 for columns
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape